# 02 — Amazon EDA

**입력**: `data/silver/amazon/amazon_reviews_lemmatized.csv`, `amazon_items_processed.csv`, `skinsort_processed.csv`  
(01_amazon_preprocessing.ipynb 실행 후 생성됨)

**범위**: 카테고리 분포 / 브랜드별 리뷰·평점 / 가격 분포 / 워드클라우드 / Skinsort 성분·사용후 분석

In [ ]:
import sys
from pathlib import Path
REPO_ROOT = next(p for p in [Path.cwd().resolve()] + list(Path.cwd().resolve().parents)
                 if (p / '.git').is_dir())
sys.path.insert(0, str(REPO_ROOT / 'src'))
from util.repo_paths import SILVER_AMAZON

In [ ]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import ast
import datetime
import re
import string
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer

from sklearn.preprocessing import StandardScaler
from yellowbrick.cluster import KElbowVisualizer
from yellowbrick.cluster import intercluster_distance
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# silver bridge — 01_amazon_preprocessing.ipynb 이 먼저 실행되어야 함
amazon_df = pd.read_csv(SILVER_AMAZON / 'amazon_reviews_lemmatized.csv')
amazon_items_df = pd.read_csv(SILVER_AMAZON / 'amazon_items_processed.csv')
skinsort_copy = pd.read_csv(SILVER_AMAZON / 'skinsort_processed.csv')

### Visualization

In [ ]:
# 팔레트 색상 테스트
from matplotlib.colors import LinearSegmentedColormap
from itertools import cycle

beauty_palette = ['#C58A63', '#F8E9B5', '#E7BCA9', '#EEA77E']
market_palette = ["#003f5c", "#214876", "#4e4c8a", "#7e4a93", "#ae448d", "#d63e7a", "#f1465d", "#fd6036"]
market_palette =  [color for color, _ in zip(cycle(beauty_palette), range(len(market_palette)))]

skin_palette = ['#8C9DB7', '#E3E9F2', '#FD6036','#F6DDD3', '#525252', '#0E1012']
custom_cmap = LinearSegmentedColormap.from_list("custom_palette", market_palette)

# sns.palplot(beauty_palette)
# plt.title("Palette Test")
# plt.show()

# sns.palplot(market_palette)
# plt.title("Palette Test")
# plt.show()

# sns.palplot(skin_palette)
# plt.title("Palette Test")
# plt.show()

In [ ]:
# K-beauty 하위 카테고리 별 개수
ak_category_count = pd.DataFrame(amazon_df.Amazon_Category.value_counts())
ak_category_count.reset_index(inplace=True)
ak_category_count.columns = ['Amazon_Category', 'Count']
print(ak_category_count.head())

# K-beauty 아이템 별 개수
category_count = pd.DataFrame(amazon_df.Sub_Category_Name.value_counts())
category_count.reset_index(inplace=True)
category_count.columns = ['Sub_Category', 'Count']
print(category_count.head())

In [ ]:
# K-beauty 브랜드 별 개수
amazon_df.brand.unique()

# 브랜드 별 리뷰 개수
k_brand_count = pd.DataFrame(amazon_df.brand.value_counts())
k_brand_count.reset_index(inplace=True)
k_brand_count

In [ ]:
# 가격 분포

# 1. 가격(price) 및 평균 별점(total_star_mean) 분포 분석
plt.figure(figsize=(14, 6))

# 가격 분포
plt.subplot(1, 2, 1)
sns.histplot(amazon_items_df['price'].dropna(), bins=32, kde=True, color='#E7BCA9', alpha=0.7)
plt.title('Price Distribution')
plt.xlabel('Price')
plt.ylabel('Frequency')

# 평균 별점 분포
plt.subplot(1, 2, 2)
sns.histplot(amazon_items_df['total_star_mean'], bins=20, kde=True, color='#E7BCA9', alpha=0.7)
plt.title('Average Star Rating Distribution')
plt.xlabel('Average Star Rating')
plt.ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# 전체 가격대별 리뷰수
price_reviews_cnt = amazon_df.groupby(['price'])['review_content'].count().reset_index()

plt.figure(figsize=(20, 6))
sns.histplot(data=price_reviews_cnt, x='price', bins=25, kde=True, alpha=0.7, color='#8C9DB7')
plt.title('Number of Review by price')
plt.xlabel('price')
plt.ylabel('Number of Review Content')
plt.xticks(rotation=90)
plt.show()

In [ ]:
category_dr_counts = amazon_items_df[amazon_items_df['brand']=='Dr.Jart+']
category_cs_counts = amazon_items_df[amazon_items_df['brand']=='COSRX']
category_if_counts = amazon_items_df[amazon_items_df['brand']=="I'm from"]
category_bj_counts = amazon_items_df[amazon_items_df['brand']=='Beauty of Joseon']
category_pu_counts = amazon_items_df[amazon_items_df['brand']=='PURITO']

plt.figure(figsize=(20, 6))

# 브랜드별 제품 수 분포
plt.subplot(1, 5, 1)
category_dr_counts['category'].value_counts().plot(kind='bar', alpha=0.7, color='#FF9960')
plt.title('Dr.Jart+ : Number of Products by Category')
plt.xlabel('Category')
plt.ylabel('Number of Products')
plt.xticks(rotation=45, ha='right')
plt.subplot(1, 5, 2)
category_cs_counts['category'].value_counts().plot(kind='bar', alpha=0.7, color='#FF9960')
plt.title('COSRX : Number of Products by Category')
plt.xlabel('Category')
plt.ylabel('Number of Products')
plt.xticks(rotation=45, ha='right')
plt.subplot(1, 5, 3)
category_if_counts['category'].value_counts().plot(kind='bar', alpha=0.7, color='#FF9960')
plt.title("I'm from : Number of Products by Category")
plt.xlabel('Category')
plt.ylabel('Number of Products')
plt.xticks(rotation=45, ha='right')
plt.subplot(1, 5, 4)
category_bj_counts['category'].value_counts().plot(kind='bar', alpha=0.7, color='#FF9960')
plt.title('Beauty of Joseon : Number of Products by Category')
plt.xlabel('Category')
plt.ylabel('Number of Products')
plt.xticks(rotation=45, ha='right')
plt.subplot(1, 5, 5)
category_pu_counts['category'].value_counts().plot(kind='bar', alpha=0.7, color='#FF9960')
plt.title('PURITO : Number of Products by Category')
plt.xlabel('Category')
plt.ylabel('Number of Products')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# 브랜드별 가격 분포

plt.figure(figsize=(20, 6))

plt.subplot(1, 5, 1)
sns.histplot(category_dr_counts['price'].dropna(), bins=29, kde=True, alpha=0.7, color='#8C9DB7')
plt.title('Dr.Jart+ : Price Distribution')
plt.xlabel('Price')
plt.ylabel('Number of Products')
plt.xticks(rotation=45, ha='right')

plt.subplot(1, 5, 2)
sns.histplot(category_cs_counts['price'].dropna(), bins=29, kde=True, alpha=0.7, color='#8C9DB7')
plt.title('COSRX : Price Distribution')
plt.xlabel('Price')
plt.ylabel('Number of Products')
plt.xticks(rotation=45, ha='right')

plt.subplot(1, 5, 3)
sns.histplot(category_if_counts['price'].dropna(), bins=29, kde=True, alpha=0.7, color='#8C9DB7')
plt.title("I'm from : Price Distribution")
plt.xlabel('Price')
plt.ylabel('Number of Products')
plt.xticks(rotation=45, ha='right')

plt.subplot(1, 5, 4)
sns.histplot(category_bj_counts['price'].dropna(), bins=29, kde=True, alpha=0.7, color='#8C9DB7')
plt.title('Beauty of Joseon : Price Distribution')
plt.xlabel('Price')
plt.ylabel('Number of Products')
plt.xticks(rotation=45, ha='right')

plt.subplot(1, 5, 5)
sns.histplot(category_pu_counts['price'].dropna(), bins=29, kde=True, alpha=0.7, color='#8C9DB7')
plt.title('PURITO : Price Distribution')
plt.xlabel('Price')
plt.ylabel('Number of Products')
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.show()

In [ ]:
# 브랜드별 서브카테고리별 리뷰수
brand_reviews_cnt = amazon_df.groupby(['brand','Sub_Category_Name'])['review_content'].count().reset_index()

brand_reviews_cnt_dr = brand_reviews_cnt[brand_reviews_cnt['brand']=='Dr.Jart+'].reset_index()
brand_reviews_cnt_cs = brand_reviews_cnt[brand_reviews_cnt['brand']=='COSRX'].reset_index()
brand_reviews_cnt_if = brand_reviews_cnt[brand_reviews_cnt['brand']=="I'm from"].reset_index()
brand_reviews_cnt_bj = brand_reviews_cnt[brand_reviews_cnt['brand']=='Beauty of Joseon'].reset_index()
brand_reviews_cnt_pu = brand_reviews_cnt[brand_reviews_cnt['brand']=='PURITO'].reset_index()

plt.figure(figsize=(20, 6))

plt.subplot(1, 5, 1)
sns.barplot(data=brand_reviews_cnt_dr, x='Sub_Category_Name', y='review_content' , alpha=0.7, color='#8C9DB7')
plt.title('Dr.Jart+ : Number of Review by Sub Category')
plt.xlabel('Sub Category')
plt.ylabel('Number of Review Content')
plt.xticks(rotation=90)

plt.subplot(1, 5, 2)
sns.barplot(data=brand_reviews_cnt_cs, x='Sub_Category_Name', y='review_content' , alpha=0.7, color='#8C9DB7')
plt.title('COSRX : Number of Review by Sub Category')
plt.xlabel('Sub Category')
plt.ylabel('Number of Review Content')
plt.xticks(rotation=90)

plt.subplot(1, 5, 3)
sns.barplot(data=brand_reviews_cnt_if, x='Sub_Category_Name', y='review_content' , alpha=0.7, color='#8C9DB7')
plt.title("I'm from : Number of Review by Sub Category")
plt.xlabel('Sub Category')
plt.ylabel('Number of Review Content')
plt.xticks(rotation=90)

plt.subplot(1, 5, 4)
sns.barplot(data=brand_reviews_cnt_bj, x='Sub_Category_Name', y='review_content' , alpha=0.7, color='#8C9DB7')
plt.title('Beauty of Joseon : Number of Review by Sub Category')
plt.xlabel('Sub Category')
plt.ylabel('Number of Review Content')
plt.xticks(rotation=90)

plt.subplot(1, 5, 5)
sns.barplot(data=brand_reviews_cnt_pu, x='Sub_Category_Name', y='review_content' , alpha=0.7, color='#8C9DB7')
plt.title('PURITO : Number of Review by Sub Category')
plt.xlabel('Sub Category')
plt.ylabel('Number of Review Content')
plt.xticks(rotation=90)

plt.tight_layout()
plt.show()

In [ ]:
# 브랜드 가격별 리뷰수
brand_price_reviews_cnt = amazon_df.groupby(['brand','price'])['review_content'].count().reset_index()

brand_price_reviews_cnt_dr = brand_price_reviews_cnt[brand_price_reviews_cnt['brand']=='Dr.Jart+'].reset_index()
brand_price_reviews_cnt_cs = brand_price_reviews_cnt[brand_price_reviews_cnt['brand']=='COSRX'].reset_index()
brand_price_reviews_cnt_if = brand_price_reviews_cnt[brand_price_reviews_cnt['brand']=="I'm from"].reset_index()
brand_price_reviews_cnt_bj = brand_price_reviews_cnt[brand_price_reviews_cnt['brand']=='Beauty of Joseon'].reset_index()
brand_price_reviews_cnt_pu = brand_price_reviews_cnt[brand_price_reviews_cnt['brand']=='PURITO'].reset_index()

plt.figure(figsize=(20, 6))

plt.subplot(1, 5, 1)
sns.histplot(data=brand_price_reviews_cnt_dr, x='price', bins=20, kde=True ,alpha=0.7, color='#8C9DB7')
plt.title('Dr.Jart+ : Number of Review by price')
plt.xlabel('price')
plt.ylabel('Number of Review Content')
plt.xticks(rotation=90)

plt.subplot(1, 5, 2)
sns.histplot(data=brand_price_reviews_cnt_cs, x='price', bins=20, kde=True , alpha=0.7, color='#8C9DB7')
plt.title('COSRX : Number of Review by price')
plt.xlabel('price')
plt.ylabel('Number of Review Content')
plt.xticks(rotation=90)

plt.subplot(1, 5, 3)
sns.histplot(data=brand_price_reviews_cnt_if, x='price', bins=20, kde=True  , alpha=0.7, color='#8C9DB7')
plt.title("I'm from : Number of Review by price")
plt.xlabel('price')
plt.ylabel('Number of Review Content')
plt.xticks(rotation=90)

plt.subplot(1, 5, 4)
sns.histplot(data=brand_price_reviews_cnt_bj, x='price', bins=20, kde=True  , alpha=0.7, color='#8C9DB7')
plt.title('Beauty of Joseon : Number of Review by price')
plt.xlabel('price')
plt.ylabel('Number of Review Content')
plt.xticks(rotation=90)

plt.subplot(1, 5, 5)
sns.histplot(data=brand_price_reviews_cnt_pu, x='price', bins=20, kde=True  , alpha=0.7, color='#8C9DB7')
plt.title('PURITO : Number of Review by price')
plt.xlabel('price')
plt.ylabel('Number of Review Content')
plt.xticks(rotation=90)

plt.tight_layout()
plt.show()

In [ ]:
# 브랜드별 서브카테고리별 평균 가격
brand_reviews_mean = amazon_df.groupby(['brand','Sub_Category_Name'])['price'].mean().round(2).dropna().reset_index()

brand_reviews_mean_dr = brand_reviews_mean[brand_reviews_mean['brand']=='Dr.Jart+'].reset_index()
brand_reviews_mean_cs = brand_reviews_mean[brand_reviews_mean['brand']=='COSRX'].reset_index()
brand_reviews_mean_if = brand_reviews_mean[brand_reviews_mean['brand']=="I'm from"].reset_index()
brand_reviews_mean_bj = brand_reviews_mean[brand_reviews_mean['brand']=='Beauty of Joseon'].reset_index()
brand_reviews_mean_pu = brand_reviews_mean[brand_reviews_mean['brand']=='PURITO'].reset_index()

plt.figure(figsize=(20, 6))

plt.subplot(1, 5, 1)
sns.barplot(data=brand_reviews_mean_dr, x='Sub_Category_Name', y='price' , alpha=0.7, color='#E7BCA9')
plt.title('Dr.Jart+ : Price(mean) by Sub Category')
plt.xlabel('Sub Category')
plt.ylabel('Price(mean)')
plt.xticks(rotation=90)

plt.subplot(1, 5, 2)
sns.barplot(data=brand_reviews_mean_cs, x='Sub_Category_Name', y='price' , alpha=0.7, color='#E7BCA9')
plt.title('COSRX : Price(mean) by Sub Category')
plt.xlabel('Sub Category')
plt.ylabel('Price(mean)')
plt.xticks(rotation=90)

plt.subplot(1, 5, 3)
sns.barplot(data=brand_reviews_mean_if, x='Sub_Category_Name', y='price' , alpha=0.7, color='#E7BCA9')
plt.title("I'm from : Price(mean) by Sub Category")
plt.xlabel('Sub Category')
plt.ylabel('Price(mean)')
plt.xticks(rotation=90)

plt.subplot(1, 5, 4)
sns.barplot(data=brand_reviews_mean_bj, x='Sub_Category_Name', y='price' , alpha=0.7, color='#E7BCA9')
plt.title('Beauty of Joseon : Price(mean) by Sub Category')
plt.xlabel('Sub Category')
plt.ylabel('Price(mean)')
plt.xticks(rotation=90)

plt.subplot(1, 5, 5)
sns.barplot(data=brand_reviews_mean_pu, x='Sub_Category_Name', y='price' , alpha=0.7, color='#E7BCA9')
plt.title('PURITO : Price(mean) by Sub Category')
plt.xlabel('Sub Category')
plt.ylabel('Price(mean)')
plt.xticks(rotation=90)

plt.tight_layout()
plt.show()

In [ ]:
# 기간별 리뷰수
date_reviews_cnt = amazon_df.groupby(['review_date'])['review_content'].count().reset_index()

sns.histplot(data=date_reviews_cnt, x='review_date', bins=50, kde=True, color='#C58A63')
plt.title('Number of reviews by review_content')
plt.xlabel('review_content')
plt.ylabel('Number of reviews')
plt.xticks(rotation=45, ha='right')
plt.show()

In [ ]:
# 브랜드 기간별 리뷰수
brand_date_reviews_cnt = amazon_df.groupby(['brand','review_date'])['review_content'].count().reset_index()

brand_date_reviews_cnt_dr = brand_date_reviews_cnt[brand_date_reviews_cnt['brand']=='Dr.Jart+'].reset_index()
brand_date_reviews_cnt_cs = brand_date_reviews_cnt[brand_date_reviews_cnt['brand']=='COSRX'].reset_index()
brand_date_reviews_cnt_if = brand_date_reviews_cnt[brand_date_reviews_cnt['brand']=="I'm from"].reset_index()
brand_date_reviews_cnt_bj = brand_date_reviews_cnt[brand_date_reviews_cnt['brand']=='Beauty of Joseon'].reset_index()
brand_date_reviews_cnt_pu = brand_date_reviews_cnt[brand_date_reviews_cnt['brand']=='PURITO'].reset_index()

plt.figure(figsize=(20, 6))

plt.subplot(1, 5, 1)
sns.histplot(data=brand_date_reviews_cnt_dr, x='review_date', bins=50, kde=True, color='#C58A63')
plt.title('Dr.Jart+ : Number of Review by review_date')
plt.xlabel('review_date')
plt.ylabel('Number of Review Content')
plt.xticks(rotation=45)

plt.subplot(1, 5, 2)
sns.histplot(data=brand_date_reviews_cnt_cs, x='review_date', bins=50, kde=True, color='#C58A63')
plt.title('COSRX : Number of Review by review_date')
plt.xlabel('review_date')
plt.ylabel('Number of Review Content')
plt.xticks(rotation=45)

plt.subplot(1, 5, 3)
sns.histplot(data=brand_date_reviews_cnt_if, x='review_date', bins=50, kde=True, color='#C58A63')
plt.title("I'm from : Number of Review by review_date")
plt.xlabel('review_date')
plt.ylabel('Number of Review Content')
plt.xticks(rotation=45)

plt.subplot(1, 5, 4)
sns.histplot(data=brand_date_reviews_cnt_bj, x='review_date', bins=50, kde=True, color='#C58A63')
plt.title('Beauty of Joseon : Number of Review by review_date')
plt.xlabel('review_date')
plt.ylabel('Number of Review Content')
plt.xticks(rotation=45)

plt.subplot(1, 5, 5)
sns.histplot(data=brand_date_reviews_cnt_pu, x='review_date', bins=50, kde=True, color='#C58A63')
plt.title('PURITO : Number of Review by review_date')
plt.xlabel('review_date')
plt.ylabel('Number of Review Content')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# 1. 제품 성능 심화 분석
amazon_df_copy = amazon_df.copy()
amazon_df_copy_coef = amazon_df_copy.groupby('total_star_mean')['review_content'].count().reset_index()


## (1) 베스트셀러 탐색: 카테고리별 상위 제품 특성 분석
category_best = amazon_df_copy.groupby('Amazon_Category').apply(
    lambda x: x.nsmallest(5, 'Category_Rank')
).reset_index(drop=True)

plt.figure(figsize=(10, 6))
sns.barplot(data=category_best, x='Amazon_Category', y='price', ci=None, palette='cool')
plt.title('Top Products by Category Rank (Price Comparison)')
plt.xticks(rotation=45)
plt.show()

## (2) 가격 민감도 분석: 동일 카테고리 내 가격 대비 평점
selected_category = 'Skin Care Products'
category_data = amazon_df_copy.groupby(['Amazon_Category','price'])['total_star_mean'].mean().reset_index()
category_data_coef = category_data[category_data['Amazon_Category'] == selected_category]

plt.figure(figsize=(10, 6))
sns.scatterplot(data=category_data_coef, x='price', y='total_star_mean', alpha=0.6, color='blue')
plt.title(f'Price vs Average Rating in {selected_category}')
plt.xlabel('Price')
plt.ylabel('Average Star Rating')
plt.grid(True)
plt.show()

## (3) 리뷰 개수와 평점 간 상관관계 분석
plt.figure(figsize=(10, 6))
sns.scatterplot(data=amazon_df_copy_coef, x='review_content', y='total_star_mean', alpha=0.6, color='green')
plt.title('Review Count vs Average Star Rating')
plt.xlabel('Number of Reviews')
plt.ylabel('Average Star Rating')
plt.grid(True)
plt.show()

In [ ]:
# 브랜드 분포
plt.figure(figsize=(12,8))
sns.histplot(data=amazon_items_df['brand'], bins=5, alpha=0.7, color='#FF9960', kde=True)
plt.title('Distribution of K-beauty Brand in US Amazon', fontsize=20, pad=20)

plt.xticks(rotation=65, fontsize=12)
plt.xlabel('K-beauty Brand', fontsize=14)
plt.ylabel('Items_cnt', fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:
# 제품 유형 분포
plt.figure(figsize=(24,10))
sns.histplot(data=amazon_df['Sub_Category_Name'], bins=31, alpha=0.7, color='#FF9960', kde=True)
plt.title('K-beauty Item Type in US Amazon', fontsize=20, pad=20)

plt.xticks(rotation=65, fontsize=12)
plt.xlabel('Frequency', fontsize=14)
plt.ylabel('K-beauty Category', fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:
# <제품 유형, 브랜드 별 <-> 리뷰 평점 확인>
# 1. 제품 유형 별 평균 리뷰 평점
category_mean = pd.pivot_table(amazon_df, index=['Sub_Category_Name'], values='review_rating',aggfunc='mean')
category_mean = category_mean.reset_index().sort_values(by='review_rating', ascending=False)
category_mean.head(2)

plt.figure(figsize=(14, 6))
sns.barplot(data=category_mean, x='Sub_Category_Name', y='review_rating', dodge=False, palette=market_palette[-1::-1], alpha=0.7)
plt.xticks(rotation=45, ha='right')
plt.title('Average Review Score by Item Type', pad=20)
plt.ylabel('Average Review Score')
plt.xlabel('Item Type')
plt.tight_layout()
plt.show()

In [ ]:
# 2. 브랜드별 평균 리뷰 평점
filtered_df = amazon_df[['brand', 'review_rating']].copy()

filtered_df['review_rating'] = pd.to_numeric(filtered_df['review_rating'], errors='coerce')
filtered_df.dropna(subset=['brand', 'review_rating'], inplace=True)

brand_review_score = pd.pivot_table(
    filtered_df,
    index=['brand'],
    values='review_rating',
    aggfunc='mean'
)
brand_review_score = brand_review_score.reset_index().sort_values(by='review_rating', ascending=False)
brand_review_score.reset_index(inplace=True, drop=True)
brand_review_score[:20]

brand_review_score.head(2)

plt.figure(figsize=(24, 8))
sns.barplot(data=brand_review_score, x='brand', y='review_rating', dodge=False, palette=market_palette[-1::-1], alpha=0.7)
plt.xticks(rotation=45, ha='right')
plt.title('Average Review Score by Brand', pad=20)
plt.ylabel('Average Review Score')
plt.xlabel('Brand')
plt.tight_layout()
plt.show()

In [ ]:
# 브랜드별 평균 평점 시각화
import plotly.graph_objects as go

df = brand_review_score

fig = go.Figure(data=[go.Table(
    header=dict(values=df.columns, fill_color='paleturquoise', align='center'),
    cells=dict(values=[df[col] for col in df.columns], fill_color='lavender', align='center'))
])

fig.update_layout(title='Brand Review Scores')
fig.show()

In [ ]:
# 전반적인 상관관계
ordinal_palette = [
    '#C58A63',
    '#F1D0A3',
    '#F2BFA1',
    '#F69D75',
    '#EEA77E',
    '#F89C6B',
    '#F9A67A',
    '#FDB58A'
]

plt.close()

sns.reset_defaults()
plt.rcdefaults()
plt.style.use('default')
plt.figure(figsize=(8,6))

sample_data = amazon_df.select_dtypes(include=["number"])

#compute correlation
corr_matrix = sample_data.corr()
corr_matrix

#annot=True return the correlation values
sns.heatmap(corr_matrix, cmap=ordinal_palette, annot=True)
plt.xticks(rotation=45)
plt.title('Correlation Matrix of Amazon K-beauty Dataset', fontsize=14, pad=20)
plt.tight_layout()
plt.show()

### Word Cloud

In [ ]:
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import re
import string
# # porter stemmer 초기화 및 영어 불용어 세트 생성 (캐글 참고)
# stemmer = PorterStemmer()
# # NLTK 기본 영어 불용어 세트
# stop_words = set(stopwords.words('english'))
# # K-beauty 관련 추가 불용어
# kbeauty_stopwords =  {
# "and", "beauty", "skincare", "cosmetics", "product", "products","use", "using", "from","for","im","i'm","floz",
# "skin", "care", "makeup", "mask", "sheet","best", "top", "favorite",
# "amazing", "perfect", "good", "bad", "recommend", "use", "review", "love","brand",
# "brands", "item", "items", "category", "categories", "line", "lines","formula", "formulas", "ingredient", "ingredients",
# "collection", "collections","set", "sets", "value", "values", "pack", "packs","latest", "exclusive", "limited",
# "special", "popular", "quality", "For",
# "safe", "worked", "works", " product"
# "face", "feel","really","stuff","joseon",
# "skin","used","time", "dont","makes","tried","one","skin feel","lot","trying","buy","apply","quite","way","never",
# "bought", "always","without","absolutely","might","maybe","sure","think","though",
# "getting","want","result","know", "especially","dr jart","purchase","definitely",
# "thing","started","need","type","facial","another","noticed","actually",
# "people","money","got","box","every","another","found","jart","wear","drjart","nan","1","no","non","not", "drjart","to",
# "cosrx","From" , "drjrt", "types", "of", " of", "100ml","200ml","300ml","150ml","250ml","50ml","30ml", "338","507","676","purito"
# }

# # 기존 stop_words와 K-beauty 불용어 합치기
# custom_stopwords = stop_words.union(kbeauty_stopwords)

# 텍스트 전처리
def clean_text(text):
    if isinstance(text, str):
        # 소문자로 모두 변환
        text = text.lower()
        # URL 제거
        text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
        # 마크다운 스타일 링크 제거
        text = re.sub(r'\[.*?\]\(.*?\)', '', text)
        # @ 제거
        text = re.sub(r'@\w+', '', text)
        # 구두점, 특수문자 제거
        text = text.translate(str.maketrans('', '', string.punctuation))
        return text
    else:
        return text
# 텍스트 토큰화
def tokenize_text(text):
    if isinstance(text, str):
        return word_tokenize(text)
    else:
        return text
# 불용어 제거
def remove_stopwords(tokens):
    if isinstance(tokens, list):
        return [word for word in tokens if word not in custom_stopwords]
    else:
        return tokens
# -> stemming 함수 추가
def stem_tokens(tokens):
    if isinstance(tokens, list):
        return [stemmer.stem(token) for token in tokens]
    else:
        return tokens

In [ ]:
from wordcloud import WordCloud
from collections import Counter

# 전처리된 아마존 모든 리뷰 워드 클라우드
wordcloud = WordCloud(width=800, height=400,
                      background_color='white', colormap='coolwarm').generate(' '.join(amazon_df['cleaned_review'].dropna()))
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
# plt.title("Word Cloud - Item Review")
plt.show()

amazon_df_copy = amazon_df.copy()

review_text = " ".join(review for review in amazon_df_copy['review_content'].dropna())

words = [word for word in review_text.split() if word.lower() not in stop_words]
common_words = Counter(words).most_common(30)

# 막대그래프로 표시
common_words_df = pd.DataFrame(common_words, columns=['Word', 'Count'])
plt.figure(figsize=(12, 6))
sns.barplot(data=common_words_df, x='Count', y='Word', palette='viridis')
plt.title('Top 20 Keywords in Reviews')
plt.xlabel('Count')
plt.ylabel('Word')
plt.show()

In [ ]:
# 전처리된 상품 제목 (아이템 이름) 워드 클라우드
wordcloud = WordCloud(width=800, height=400,
                      background_color='white', colormap='coolwarm').generate(' '.join(amazon_df['cleaned_title'].dropna()))
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
# plt.title("Word Cloud - Item Title")
plt.show()

### Skinsort

In [ ]:
import plotly.express as px
import plotly.io as pio

df = skinsort_copy

country_dist = df['country'].value_counts().reset_index()
country_dist.columns = ['Country', 'Count']

fig = px.pie(country_dist[:15], values='Count', names='Country', title=f"Country Distribution for Skinsort Brand Data")
fig.update_traces(hole=.3)
# imgname = "skinsort_country_distribution.png"
# fig.write_image(DATA_PATH+imgname)
fig.show()

In [ ]:
# 카테고리 분포
plt.figure(figsize=(24,10))
sns.histplot(data=skinsort_copy['type'], bins=31, alpha=0.7, color='#8C9DB7', kde=True)
plt.title('Item Type of Skin Care Brands', fontsize=20)

plt.xticks(rotation=65, fontsize=12)
plt.xlabel('Frequency', fontsize=14)
plt.ylabel('Item Type', fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:
# 한국 스킨케어 브랜드들의 제품 타입 분포
# skincare 데이터셋 활용용
query_cond = "country == 'South Korea'"
k_skin_brand = skinsort_copy.query(query_cond).reset_index()
k_skin_brand.drop(columns='index', inplace=True)
k_skin_brand.head(2)

plt.figure(figsize=(24,10))

sns.histplot(data=k_skin_brand['type'], bins=31, alpha=0.7, color='#8C9DB7', kde=True)
plt.title('Item Type of Korea Skin Care Brands', fontsize=20, pad=20)

plt.xticks(rotation=65, fontsize=12)
plt.xlabel('Frequency', fontsize=14)
plt.ylabel('Item Type', fontsize=14)

plt.tight_layout()
plt.show()

> Skinsort 한국 스킨케어 브랜드의 제품 유형 분포
- 일반적으로 알고 있는 것과 비슷함. general moisture, serum, sheet mask, toner 등 기초 제품들과 마스크팩이 많음 !

In [ ]:
k_skin_brand.head(2)

In [ ]:
# 한국 스킨케어 브랜드 성분
from wordcloud import WordCloud

wordcloud = WordCloud(width=800, height=400,
                      background_color='white', colormap='coolwarm').generate(' '.join(k_skin_brand['ingridients'].dropna()))
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
# plt.title("Word Cloud - Item Review")
plt.show()

In [ ]:
# 한국 스킨케어 브랜드 - 사용후
from wordcloud import WordCloud

wordcloud = WordCloud(width=800, height=400,
                      background_color='white', colormap='coolwarm').generate(' '.join(k_skin_brand['afterUse'].dropna()))
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
# plt.title("Word Cloud - Item Review")
plt.show()

> 사용 후 반응
- 피부 타입 개인 차에 따른 반응들이 여기도 많음. 더 기름이 심해졌다, 건조하다 등
- 간지러움, 붉어짐 등 -> 개인적인 생각이지만 너무 특이한 성분? 약초나 시카 성분 등 자연성분을 강조한다고 너무 특이한 게 들어가면 개인 차를 많이 타는 것 같음

In [ ]:

# # pay_amont 데이터 추출
# light_group_pay = df[df['player_segment3'] == 'low']['pay_amont'].dropna()
# heavy_group_pay = df[df['player_segment3'] == 'high']['pay_amont'].dropna()

# # 평균 계산
# light_mean = light_group_pay.mean()
# print("light_mean", light_mean)
# heavy_mean = heavy_group_pay.mean()
# print("heavy_mean", heavy_mean)

# # 데이터프레임 생성
# mean_data = pd.DataFrame({
#     'Group': ['Low', 'High'],
#     'Mean_Pay_Amount': [light_mean, heavy_mean]
# })

# # 시각화
# plt.figure(figsize=(8, 10))
# sns.barplot(data=mean_data, x='Group', y='Mean_Pay_Amount', palette='viridis')

# # 그래프 제목과 레이블 추가
# plt.title('Mean Pay Amount by Player Segment', fontsize=18)
# plt.xlabel('Player Segment (instance_Dungeon_enter_cnt)', fontsize=18)
# plt.ylabel('Pay Amount', fontsize=18)
# plt.xticks(fontsize=14)
# plt.yticks(fontsize=14)

# # p-value 표시
# t_stat, p_value = stats.ttest_ind(light_group_pay, heavy_group_pay, equal_var=False)
# print(t_stat, p_value)

# plt.tight_layout()
# plt.show()


In [ ]:
# # 범주형 변수 그룹 간 차이
# from scipy.stats import chi2_contingency

# behavior_cols = ['job', 'action_type', 'channel',  'main_purchase_category']

# results = []

# for column in behavior_cols:

#     contingency_table = pd.crosstab(df['player_segment3'], df[column])
#     chi2, p, dof, expected = chi2_contingency(contingency_table)
#     results.append({'behavior_column': column, 'chi2_stat': chi2, 'p_value': p, 'degrees_of_freedom': dof})

# results_df = pd.DataFrame(results)

# print(results_df)

> 유저 세그먼트화 및 차이 분석 결과
- 현재 데이터 안에서 통계적으로 유의미한 차이를 보이는 결과는 없음.
- 그래도, 한 세트를 건진다면 던전 입장 횟수 (low, high) & action type 은 눈여겨볼만 함.
- p-value 는 유의하지 않으나 유일하게 0.1 대까지 p-value 가 내려간 조합이었음.
    - 많이/적게 입장한 그룹이 주로 어떤 액션을 했는지?
        - 적게 입장 -> 정말 레이드라면 이전에 팀에서 나온 의견 맞을 가능성도 있음.
        - 오히려, 많이 -> fishing, pvp
- 이하 참고
-     behavior_column  chi2_stat   p_value  degrees_of_freedom
- 0                     job   1.389877  0.845952                   4
- 1             action_type   6.768321  0.148650                   4
- 2                 channel   2.003753  0.367190                   2
- 3  main_purchase_category   0.958777  0.995466                   7

In [ ]:
# # action_type과 player_segment3의 교차 테이블 계산
# action_type_counts = pd.crosstab(df['player_segment3'], df['action_type'])

# # 세그먼트별 데이터 추출 및 정렬
# high_group = action_type_counts.loc['high'].reset_index()
# low_group = action_type_counts.loc['low'].reset_index()

# high_group.columns = ['Action Type', 'Count']
# low_group.columns = ['Action Type', 'Count']

# # Count 기준으로 정렬
# high_group = high_group.sort_values(by='Count', ascending=False).reset_index(drop=True)
# low_group = low_group.sort_values(by='Count', ascending=False).reset_index(drop=True)

# # 고정된 컬러 맵 생성 (action_type 별 색상 동일화)
# unique_action_types = sorted(set(high_group['Action Type']).union(set(low_group['Action Type'])))
# color_palette = sns.color_palette("tab10", len(unique_action_types))
# color_mapping = {action: color for action, color in zip(unique_action_types, color_palette)}

# # 그래프 생성
# fig, axes = plt.subplots(1, 2, figsize=(14, 10), sharey=True)

# # High 그룹
# sns.barplot(
#     data=high_group,
#     x='Action Type',
#     y='Count',
#     ax=axes[0],
#     palette=[color_mapping[action] for action in high_group['Action Type']]
# )
# axes[0].set_title('Player Segment (High)', fontsize=18)
# axes[0].set_xlabel('Action Type', fontsize=14)
# axes[0].set_ylabel('Count', fontsize=14)
# axes[0].tick_params(axis='x', rotation=45, labelsize=14)

# # Low 그룹
# sns.barplot(
#     data=low_group,
#     x='Action Type',
#     y='Count',
#     ax=axes[1],
#     palette=[color_mapping[action] for action in low_group['Action Type']]
# )
# axes[1].set_title('Player Segment (Low)', fontsize=18)
# axes[1].set_xlabel('Action Type', fontsize=14)
# axes[1].set_ylabel('')  # 공유 Y축 사용
# axes[1].tick_params(axis='x', rotation=45, labelsize=14)

# # 전체 레이아웃 정리
# plt.tight_layout()
# plt.show()
